# Week 5 Lab — One model of each shape

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prof-tcsmith/6564F26-DATA/blob/main/week-05-transformers-llms/lab/lab-slm-by-shape.ipynb)

The slide *Today's small language models, by shape* said that the shape is the part that lasts. This lab puts
one model of each shape in your hands: an **encoder** that fills a blank and one that turns a text into a vector,
a **decoder** that continues text and one that answers, and an **encoder–decoder** that translates and summarises.
About 40 minutes. Every station has a cell that runs as written and a **YOUR TURN** cell where you change the input.

**Two ways to run it**

- **Google Colab** (recommended for the lab). Click the badge. Then *Runtime → Change runtime type → T4 GPU → Save*.
  If Colab offers no GPU right now, stay on CPU: everything still works, Station 3 is just slower.
- **Your own machine.** Follow the [getting-started guide](https://github.com/prof-tcsmith/6564F26-DATA/blob/main/docs/getting-started-local.md) once (Miniconda, VS Code, your GPU), then open
  this notebook in VS Code and select the `Python (ism6564)` kernel. The models total about 2.7 GB: run the first
  two cells **at home**, not on classroom Wi-Fi.

Fill in the **results sheet** at the bottom as you go. The discussion at the end of the lab works from it.

In [ ]:
# Setup — runs in Google Colab and in a local environment.
import os, sys, time, gc, textwrap

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab ships torch. Pin the two libraries this lab leans on (quiet, about 30 s).
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers>=4.45", "sentence-transformers>=3.0", "accelerate", "sentencepiece"], check=True)

import torch, transformers, pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

transformers.logging.set_verbosity_error()
from huggingface_hub.utils import logging as hub_logging
hub_logging.set_verbosity_error()
os.environ["TOKENIZERS_PARALLELISM"] = "false"
pd.set_option("display.width", 120)

# The same three-way choice every course notebook makes: NVIDIA GPU, Apple GPU, or CPU.
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
where = "Google Colab" if IN_COLAB else "a local environment"
gpu = f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""
print(f"Python {sys.version.split()[0]} · torch {torch.__version__} · transformers {transformers.__version__}")
print(f"Running in {where} on {DEVICE.upper()}{gpu}")
if DEVICE == "cpu":
    print("No accelerator found. Everything still runs; Station 3 will be slower (a few tokens per second).")

## Step 0 — Fetch the models

Five checkpoints, all ungated: no Hugging Face account or token is needed. They land in the Hugging Face cache and
are read from there afterwards. On Colab the cache is wiped when the session ends, so this cell runs again next time
(about a minute on Colab's connection). On your own machine it runs once.

| Repo | Shape | Job in this lab | Download |
|---|---|---|---|
| `bert-base-uncased` | encoder-only | fills a blank | 0.44 GB |
| `sentence-transformers/all-MiniLM-L6-v2` | encoder-only | one vector per text | 0.09 GB |
| `Qwen/Qwen2.5-0.5B` | decoder-only, **base** | continues text | 0.99 GB |
| `Qwen/Qwen2.5-0.5B-Instruct` | decoder-only, **instruct** | answers | 0.99 GB |
| `t5-small` | encoder–decoder | translates, summarises | 0.24 GB |

In [ ]:
from huggingface_hub import snapshot_download

MODELS = {  # repo id -> shape and job
    "bert-base-uncased":                      "encoder-only     · fills a blank",
    "sentence-transformers/all-MiniLM-L6-v2": "encoder-only     · one vector per text",
    "Qwen/Qwen2.5-0.5B":                      "decoder-only     · base: continues text",
    "Qwen/Qwen2.5-0.5B-Instruct":             "decoder-only     · instruct: answers",
    "t5-small":                               "encoder–decoder  · translates, summarises",
}
# Older repos carry the same weights in several formats; we only need the safetensors copy.
SKIP = ["*.bin", "*.h5", "*.msgpack", "*.ot", "*.tflite", "onnx/*", "openvino/*", "coreml/*"]

def folder_gb(path):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs) / 1e9

t_all = time.time()
for repo, role in MODELS.items():
    t = time.time()
    path = snapshot_download(repo, ignore_patterns=SKIP)
    print(f"{repo:42s} {role:42s} {folder_gb(path):4.2f} GB  {time.time() - t:5.1f} s")
print(f"\nAll five ready in {time.time() - t_all:.0f} s. Cache: {os.path.dirname(os.path.dirname(path))}")

## Station 1 — Encoder-only: fill a blank (`bert-base-uncased`)

BERT was trained by hiding words and restoring them. The claim on slide 26 was that it reads **both** directions.
Here is the test: two sentences that are **identical up to and including the blank** and differ only after it.
A model that could see only the words to the left would have to answer both the same way.

In [ ]:
fill = pipeline("fill-mask", model="bert-base-uncased", device=DEVICE)

def show_fill(sentences, top_k=4):
    for s in sentences:
        guesses = "   ".join(f"{g['token_str']} ({g['score']:.2f})" for g in fill(s, top_k=top_k))
        print(f"{s}\n    → {guesses}\n")

show_fill(["Please send me a [MASK] , the jacket is too small .",
           "Please send me a [MASK] , I lost my receipt ."])

In [ ]:
# YOUR TURN — write a pair that is identical up to and including [MASK] and differs only AFTER it.
# If the answers differ, the model used the words to the RIGHT of the blank. Try to make them differ a lot.
my_pair = ["We need a bigger [MASK] for the party , there are forty guests .",
           "We need a bigger [MASK] for the party , the cake will not fit ."]
show_fill(my_pair)

**Record on the results sheet:** your pair, the top guess for each, and whether the words after the blank changed the answer.
Notice what BERT does *not* do here: it does not write anything. It fills a blank someone else chose. That is what an
encoder is for.

## Station 2 — Encoder-only: one vector per text (`all-MiniLM-L6-v2`)

Nobody ships blank-filling. What gets shipped is the vector. This encoder returns one 384-number vector for a whole
text, and texts that mean the same thing get similar vectors **even when they share no words**. That is slide 27,
and it is the engine of Week 7's retrieval systems. The table is cosine similarity, exactly Week 2's measure.

In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

def similarity_table(texts):
    E = embedder.encode(texts, normalize_embeddings=True)   # one 384-number vector per text
    S = E @ E.T                                              # cosine similarity, because the vectors are normalised
    labels = [f"T{i + 1}" for i in range(len(texts))]
    for l, t in zip(labels, texts):
        print(f"{l}  {t}")
    display(pd.DataFrame(S, index=labels, columns=labels).round(2))
    for i, l in enumerate(labels):                           # each text's nearest neighbour
        j = max((k for k in range(len(texts)) if k != i), key=lambda k: S[i, k])
        print(f"{l} is closest to {labels[j]}  ({S[i, j]:.2f})")

tickets = ["My package never arrived.",
           "The parcel has not been delivered yet.",
           "I was charged twice for one order.",
           "How do I reset my password?",
           "The jacket arrived damaged and I want a refund."]
similarity_table(tickets)

In [ ]:
# YOUR TURN — five short texts of your own. Include two that mean the same thing in different words,
# and one that has nothing to do with the others. Does the table find them?
my_texts = ["Stocks fell sharply after the rate decision.",
            "Markets dropped when the central bank raised rates.",
            "The new stadium opens next spring.",
            "Our flight was delayed by three hours.",
            "The plane left three hours late."]
similarity_table(my_texts)

**Record:** which pair the model put closest, whether it shared words, and which text was the odd one out.
This model has 22.7 million learned numbers, about a twentieth of the chat model in the next station. That is why it
can be run over millions of documents.

## Station 3 — Decoder-only: base vs instruct, temperature, speed (`Qwen2.5-0.5B`)

A decoder is trained to predict the next token. The **base** checkpoint does exactly that: hand it a customer's message
and it writes more of the customer's message. The **instruct** checkpoint is the same 494 million numbers after a second,
smaller stage of training on request–answer pairs (slide 32). Handed the same message through its chat template, it answers.

In [ ]:
def load_decoder(repo):
    tok = AutoTokenizer.from_pretrained(repo)
    model = AutoModelForCausalLM.from_pretrained(repo).float().to(DEVICE).eval()   # float32 on every version and device
    return tok, model

def generate(tok, model, text, chat=False, temperature=0.0, max_new_tokens=60, seed=0):
    """Greedy when temperature is 0, otherwise sample at that temperature. Returns (text, n_new_tokens, seconds)."""
    if chat:  # the instruct model was tuned on this exact string format; apply_chat_template writes it
        text = tok.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(DEVICE)
    kw = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id or tok.eos_token_id)
    if temperature == 0:
        kw.update(do_sample=False)                                   # greedy: the checkpoint's own sampling defaults are overridden
    else:
        kw.update(do_sample=True, temperature=temperature, top_p=1.0, top_k=0)
    torch.manual_seed(seed)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**ids, **kw)
    seconds = time.time() - t0
    new = out[0, ids["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip(), len(new), seconds

MESSAGE = "I bought a jacket last week and I want to return it."

tok_b, base = load_decoder("Qwen/Qwen2.5-0.5B")            # base: trained only to continue text
tok_i, inst = load_decoder("Qwen/Qwen2.5-0.5B-Instruct")   # the same model after instruction tuning

print("BASE — handed the message as plain text, it continues it:\n")
print(generate(tok_b, base, MESSAGE)[0])
print("\nINSTRUCT — handed the message through its chat template, it answers:\n")
print(generate(tok_i, inst, MESSAGE, chat=True)[0])

del base; gc.collect()                                     # free about 2 GB; the rest of the station uses the instruct model
if DEVICE == "cuda":
    torch.cuda.empty_cache()

**Temperature.** Greedy decoding takes the most likely token every time. Sampling at a temperature flattens or sharpens the
distribution before drawing from it: low temperatures stay close to greedy, high temperatures wander. Same question, three settings.

In [ ]:
QUESTION = "In two sentences, explain to a customer why a refund takes five business days."
for T in (0.0, 0.7, 1.5):
    answer, n, s = generate(tok_i, inst, QUESTION, chat=True, temperature=T, max_new_tokens=60)
    label = "greedy (temperature 0)" if T == 0 else f"temperature {T}"
    print(f"── {label} ──\n{answer}\n")

**Speed.** Generation is the Week 4 loop, one token at a time, so the number that matters is tokens per second on *your*
machine. Compare with a neighbour on a different machine or runtime. Assignment 5 measures this properly, with the
KV cache and precision in play (slides 35 and 36).

In [ ]:
answer, n, s = generate(tok_i, inst, "Explain in three sentences why a return might be refused.", chat=True, max_new_tokens=48)
print(f"{n} tokens in {s:.1f} s  →  {n / s:.1f} tokens per second on {DEVICE.upper()}, float32")

In [ ]:
# YOUR TURN — your own question, greedy and then sampled. Try one where the model has to KNOW something.
my_question = "Which is heavier, a kilogram of feathers or a kilogram of steel? Answer in one sentence."
for T in (0.0, 1.0):
    print(f"── temperature {T} ──\n{generate(tok_i, inst, my_question, chat=True, temperature=T)[0]}\n")

**Record:** what the base model did with the message, what the instruct model did, how the three temperatures differed,
and your tokens per second with the device it ran on. If your fact question came back wrong, that is data too:
494 million learned numbers is a small model, and slide 37 measures exactly this.

## Station 4 — Encoder–decoder: read everything, then write (`t5-small`)

T5 keeps both halves: an encoder reads the whole input, a decoder writes the output while looking back at it (slide 29).
Its authors phrased every job as text-to-text with **the task named in the input**. Same model, three jobs.

In [ ]:
from transformers import AutoModelForSeq2SeqLM     # "Seq2Seq": the Hugging Face name for encoder–decoder

t5_tok = AutoTokenizer.from_pretrained("t5-small")
t5 = AutoModelForSeq2SeqLM.from_pretrained("t5-small").float().to(DEVICE).eval()

LONG_TICKET = ("I ordered a blue jacket on the 3rd and it arrived on the 10th with a torn sleeve. "
               "I contacted support twice and was told a replacement would ship, but nothing has arrived. "
               "I would now like a full refund to my original card and a prepaid label to return the damaged item.")

def show_t5(prompts, max_new_tokens=48):
    for p in prompts:
        ids = t5_tok(p, return_tensors="pt").to(DEVICE)          # the encoder reads all of this first ...
        with torch.no_grad():
            out = t5.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)   # ... then the decoder writes
        text = t5_tok.decode(out[0], skip_special_tokens=True)
        head = p if len(p) <= 64 else p[:64] + "…"
        print(f"{head}\n    → {text}\n")

show_t5(["translate English to German: The package arrived damaged.",
         "translate English to French: Where is my order?",
         "summarize: " + LONG_TICKET])

In [ ]:
# YOUR TURN — the task is in the prompt. Try a language T5 was NOT trained on, and a summary of a text of your own.
show_t5(["translate English to Spanish: The package arrived damaged.",
         "summarize: " + "Replace this with three or four sentences of your own, about anything."])

**Record:** what T5 did with the language it never saw in training, and whether the summary was a summary or a copy.
The model has 60 million learned numbers; the point is the shape, not the quality.

## Station 5 — What did you load? Ask each model what it is

The last column of the slide's table was each model's own class name. Print them for the models you just used, with
their learned-number counts.

In [ ]:
loaded = {"bert-base-uncased":      fill.model,
          "all-MiniLM-L6-v2":       embedder[0].auto_model,
          "Qwen2.5-0.5B-Instruct":  inst,
          "t5-small":               t5}

print(f"{'model':24s} {'what it calls itself':30s} {'learned numbers':>16s}")
for name, m in loaded.items():
    n = sum(p.numel() for p in m.parameters())
    print(f"{name:24s} {type(m).__name__:30s} {n / 1e6:13.1f} M")

Read the suffixes: **`ForMaskedLM`** is an encoder trained by blank-filling; a bare **`BertModel`** is the same encoder
without that head, which is all an embedder needs; **`ForCausalLM`** is a decoder trained on the next token, *causal*
meaning "may only look back"; **`ForConditionalGeneration`** is an encoder–decoder. Those are the class names you will
choose between in Assignment 5.

## Results sheet

Fill this in as you go. Double-click the cell to edit it (in Colab, click the pencil), then run it to render.

| Station | Shape | Model | What you tried | What came back | Explain it in the deck's terms, one line |
|---|---|---|---|---|---|
| 1 | encoder-only | bert-base-uncased | | | |
| 2 | encoder-only | all-MiniLM-L6-v2 | | | |
| 3a | decoder-only | Qwen2.5-0.5B vs -Instruct | | | |
| 3b | decoder-only | Qwen2.5-0.5B-Instruct | temperature 0 / 0.7 / 1.5 | | |
| 3c | decoder-only | Qwen2.5-0.5B-Instruct | 48 tokens on ________ | ______ tokens/s | |
| 4 | encoder–decoder | t5-small | | | |

**Bring to the discussion**

1. Northwind wants to route two million archived tickets into six queues. Which station's model would you reach for, and why not the chat model?
2. Station 3a: same size, same architecture, opposite behaviour. What made the difference, and where on the Hub would you look to tell the two apart *before* downloading?
3. Station 4: what did T5 do with the language it was never trained on? What does that say about "the task is in the prompt"?
4. Station 3c: compare your tokens per second with a neighbour's. Name what differs between your machines, and predict how the 1.5B model in Assignment 5 will compare.

**What this sets up.** The practice assignment takes the decoder apart: tokenizer, chat template, logits, the causal mask,
the KV cache. Assignment 5 measures three decoders on your own machine and writes the recommendation. Week 7 builds a
retrieval system on exactly Station 2's model.